# Kiểm tra dữ liệu

In [1]:
import pandas as pd
from os.path import join

Data_path = join("..","Data","iris","BezdekIris.data")
cols = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]
data_raw = pd.read_csv(Data_path, sep =",", header= None, names= cols)
print(data_raw.head())

print(data_raw.info())

   sepal_length  sepal_width  petal_length  petal_width        class
0           5.1          3.5           1.4          0.2  Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa
2           4.7          3.2           1.3          0.2  Iris-setosa
3           4.6          3.1           1.5          0.2  Iris-setosa
4           5.0          3.6           1.4          0.2  Iris-setosa
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   class         150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB
None


# Tách data ra chuẩn bị cho việc train và dự đoán

In [2]:
from sklearn.model_selection import train_test_split

X = data_raw.iloc[:,:-1].values
y = data_raw.iloc[:,-1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state= 42, shuffle= True
)

data = {}  
for features, label in zip(X_train, y_train):
    if label not in data:
        data[label] = []
    data[label].append(features)

# Sử dụng hàm predict theo tham khảo

In [4]:
import numpy as np
from collections import defaultdict

# --- 1. Tách dữ liệu theo class từ train set ---
unique_classes = np.unique(y_train)
features_num = X_train.shape[1]

# Lưu feature values từng class
class_features = {c: [] for c in unique_classes}

for x, label in zip(X_train, y_train):
    class_features[label].append(x)

# --- 2. Prior P(class) ---
total_train = len(y_train)
P_class = {c: len(class_features[c]) / total_train for c in unique_classes}

# --- 3. Tính mean & variance từng class ---
class_means = {}
class_vars = {}

for c in unique_classes:
    arr = np.array(class_features[c])
    class_means[c] = np.mean(arr, axis=0)
    class_vars[c]  = np.var(arr, axis=0)

# --- 4. Gaussian PDF ---
def pdf(x, mean, var):
    var = np.maximum(var, 1e-6)  # tránh chia 0
    return 1 / np.sqrt(2 * np.pi * var) * np.exp(-(x - mean)**2 / (2*var))

# --- 5. Predict 1 sample ---
def predict(sample):
    x = np.array(sample, dtype=float)
    posteriors = {}
    for c in unique_classes:
        likelihoods = pdf(x, class_means[c], class_vars[c])
        posteriors[c] = P_class[c] * np.prod(likelihoods)
    pred_class = max(posteriors, key=posteriors.get)
    return pred_class, posteriors

# --- 6. Test trên test set ---
correct = 0
for i in range(len(X_test)):
    sample = X_test[i]
    pred, _ = predict(sample)
    if pred == y_test[i]:
        correct += 1

accuracy = correct / len(X_test)
print("Test accuracy =", accuracy)


Test accuracy = 1.0
